In [1]:
!pip -q install "syncode==0.4.16" "transformers==4.53.2"

In [2]:
from syncode import Syncode

In [3]:
FEW_SHOT_PROMPT = '''You extract two numeric operands and one arithmetic operator from the query.
Preserve the operands in their original order.
Return only two lines:
Operands: <first number> and <second number>
Operator: <one of +, -, *, />

Query: What is 327. multiplied by 11.0?
Output:
Operands: 327.0 and 11.0
Operator: *

Query: What is 2.5 plus 3.4?
Output:
Operands: 2.5 and 3.4
Operator: +

Query: What is 25.4 divided by 34.0?
Output:
Operands: 25.4 and 34.0
Operator: /

Query: What is 14.8 minus 32.0?
Output:
Operands: 14.8 and 32.0
Operator: -
'''

def build_prompt(textual_query):
    return FEW_SHOT_PROMPT + f"\nQuery: {textual_query.strip()}\nOutput:\n"

In [4]:
grammar = '''start: "Operands:" SPACE number SPACE "and" SPACE number NEWLINE "Operator:" SPACE operator
number: /-?(?:[0-9]+(?:\\.[0-9]*)?|\\.[0-9]+)/
operator: "+" | "-" | "*" | "/"
SPACE: " "
NEWLINE: "\\n"
'''

def get_answer(operator, operands):
  if operator == "+":
      final_answer = operands[0] + operands[1]
  elif operator == "-":
      final_answer = operands[0] - operands[1]
  elif operator == "*":
      final_answer = operands[0] * operands[1]
  elif operator == "/":
      final_answer = operands[0] / operands[1]
  else:
      raise ValueError("Unsupported operator")
  return float(final_answer)


syncode_model = Syncode(
    model="microsoft/phi-2", mode="grammar_strict", grammar=grammar,
    parse_output_only=True,
    quantize=True,
    device="cuda",
    parser="lalr",
    seed=0,
    max_new_tokens=48,
    do_sample=False,
    num_beams=1,
    use_cache=True,
)

[2026-10-27 21:53:43,006-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[2026-10-27 21:53:55,970-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
def run_syncode(textual_query):
    generated_text = syncode_model.infer(build_prompt(textual_query))[0].strip()
    lines = [line.strip() for line in generated_text.splitlines() if line.strip()]

    first_operand, second_operand = lines[0].removeprefix("Operands:").strip().split(" and ")
    operands = [float(first_operand), float(second_operand)]
    operator = lines[1].removeprefix("Operator:").strip()

    final_answer = get_answer(operator, operands)

    return operands, operator, final_answer

In [6]:
print("\nCase from instruction:")
print("** if falied, there should print less than 4 lines **\n")
test_case = [
    ("What is 327. multiplied by 11.0?", ([327.0, 11.0], "*", 3597.0)),
    ("What is 45.1 plus 23.54?", ([45.1, 23.54], "+", 68.64)),
    ("What is 120.4 divided by 4.0?", ([120.4, 4.0], "/", 30.1)),
    ("What is 98.5 minus 12.25?", ([98.5, 12.25], "-", 86.25)),
]

for q, ref in test_case:
    actual = run_syncode(q)
    print(actual)
    assert actual[0] == ref[0]
    assert actual[1] == ref[1]
    assert abs(actual[2] - ref[2]) < 1e-9
    assert isinstance(actual[2], float)
    print(q, "->", actual)

print("\nExtra\n")
new_case = [
    ("What is 7.25 plus 2.75?", ([7.25, 2.75], "+", 10.0)),
    ("What is 50.0 minus 12.5?", ([50.0, 12.5], "-", 37.5)),
    ("What is 3.2 multiplied by 4.5?", ([3.2, 4.5], "*", 14.4)),
    ("What is 81.0 divided by 2.5?", ([81.0, 2.5], "/", 32.4)),
]

for q, ref in new_case:
    actual = run_syncode(q)
    print(actual)
    assert actual[0] == ref[0]
    assert actual[1] == ref[1]
    assert abs(actual[2] - ref[2]) < 1e-9
    assert isinstance(actual[2], float)
    print(q, "->", actual)


Case from instruction:
** if falied, there should print less than 4 lines **



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([327.0, 11.0], '*', 3597.0)
What is 327. multiplied by 11.0? -> ([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)
What is 45.1 plus 23.54? -> ([45.1, 23.54], '+', 68.64)
([120.4, 4.0], '/', 30.1)
What is 120.4 divided by 4.0? -> ([120.4, 4.0], '/', 30.1)
([98.5, 12.25], '-', 86.25)
What is 98.5 minus 12.25? -> ([98.5, 12.25], '-', 86.25)

Extra

([7.25, 2.75], '+', 10.0)
What is 7.25 plus 2.75? -> ([7.25, 2.75], '+', 10.0)
([50.0, 12.5], '-', 37.5)
What is 50.0 minus 12.5? -> ([50.0, 12.5], '-', 37.5)
([3.2, 4.5], '*', 14.4)
What is 3.2 multiplied by 4.5? -> ([3.2, 4.5], '*', 14.4)
([81.0, 2.5], '/', 32.4)
What is 81.0 divided by 2.5? -> ([81.0, 2.5], '/', 32.4)
